# History Matching: Manual Workflow

This notebook shows how to use history matching in **manual mode**, where you control each
iteration individually. Manual mode lets you:

- Inspect results before committing to the next iteration
- Change feature selection or emulator type between iterations
- Evaluate emulator diagnostics at each step
- Gain a detailed understanding of what the algorithm is doing

For the fully automated one-call workflow, see `01_basic_workflow.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import history_matching as hm
from model import SIR, generate_observed_data

%matplotlib inline

## Generate Synthetic Observed Data

We define true parameter values and generate synthetic incidence data that we will
try to recover through history matching.

In [ ]:
beta_true  = 1.3
gamma_true = 0.5
population_size    = 10_000
n_seed_infections  = 100

incidence_obs, true_model = generate_observed_data(
    beta_true=beta_true,
    gamma_true=gamma_true,
    population_size=population_size,
    n_seed_infections=n_seed_infections,
    seed=0,
)

data = incidence_obs.to_frame()  # DataFrame with column 'incidence', index 'day'
data.head()

In [ ]:
data.plot(style="o")
plt.title("Observed daily incidence")
plt.ylabel("New cases")
plt.xlabel("Day")
plt.grid(linestyle=":")
plt.show()

## Define Parameter Space and Observations

In [ ]:
parameter_bounds = {
    "beta":  (0.1, 5.0),
    "gamma": (0.1, 1.0),
}

# One observation per day of the incidence time series
observations_dict = {
    f"incidence_{day}": (value, 10.0)
    for day, value in enumerate(data["incidence"])
}

print(f"Parameter space: {parameter_bounds}")
print(f"Observations: {len(observations_dict)} features (one per day)")

## Define Simulation Function

The simulation function must accept a DataFrame of parameter samples and return a DataFrame
of outputs. Column names must match the observation keys.

In [ ]:
def sir_model_wrapper(points: pd.DataFrame) -> pd.DataFrame:
    """Run SIR for each row and return incidence time series."""
    results = []
    for parameters in points.itertuples():
        model = SIR(
            beta=parameters.beta,
            gamma=parameters.gamma,
            s0=population_size - n_seed_infections,
            i0=n_seed_infections,
        )
        results.append(pd.Series(model.get_incidence()))
    return pd.concat(results, axis=1).T.add_prefix("incidence_").reset_index(drop=True)

In [ ]:
# Test with a few random points to confirm the wrapper works
np.random.seed(42)
test_points = pd.DataFrame({
    "beta":  np.random.uniform(0.1, 5.0, 5),
    "gamma": np.random.uniform(0.1, 1.0, 5),
})
test_results = sir_model_wrapper(test_points)
print(f"Output shape: {test_results.shape}  (5 runs x {test_results.shape[1]} days)")

## Configure the History Matching Engine

We use the builder to configure the engine with a manually chosen feature for the first
iteration. Feature selection can be updated before each subsequent iteration.

In [ ]:
builder = hm.HistoryMatchingBuilder.from_data(parameter_bounds, observations_dict)
engine = (builder
    .with_sampling_strategy("lhs")
    .with_feature_selection(["incidence_11"])  # one feature for iteration 1
    .with_emulator_type("gpr")
    .with_samples_per_iteration(100)
    .with_max_iterations(3)
    .with_implausibility_threshold(3.0)
    .with_random_seed(42)
    .build()
)

engine.set_simulation_function(sir_model_wrapper)

print(f"Engine ready.")
print(f"  Feature for iteration 1: {engine._feature_selection_strategy.selected_features}")
print(f"  Samples per iteration:   {engine._n_samples}")

## Iteration 1

Run the first iteration, inspect the results, then commit.

In [ ]:
print("Running iteration 1...")
result_1 = engine.step()

print(f"  Samples generated:  {len(result_1.samples)}")
print(f"  Features emulated:  {result_1.selected_features}")
print(f"  Emulators trained:  {len(result_1.emulators)}")

### Inspect Emulator Performance

In [ ]:
emulator_1 = result_1.get_emulator_for_feature("incidence_11")
print(f"Emulator type: {type(emulator_1).__name__}")
print(f"Training complete: {emulator_1.training_complete}")
emulator_1.info()

In [ ]:
emulator_1.test()
emulator_1.plot_diagnostics()

### Inspect Parameter Space After Iteration 1

In [ ]:
samples_1 = result_1.samples
pending_2 = engine.get_pending_next_samples()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.scatter(samples_1["beta"], samples_1["gamma"], alpha=0.6, s=20, color="gray", label="Iteration 1 samples")
ax.scatter(pending_2["beta"],  pending_2["gamma"],  alpha=0.6, s=20, color="green", label="Pending for iteration 2")
ax.axvline(beta_true,  color="red", linestyle="--", linewidth=2, label=f"True beta={beta_true}")
ax.axhline(gamma_true, color="red", linestyle="--", linewidth=2, label=f"True gamma={gamma_true}")
ax.set_xlabel("beta")
ax.set_ylabel("gamma")
ax.set_title("Parameter space after iteration 1")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
sims = sir_model_wrapper(samples_1)
sims.T.plot(color="gray", alpha=0.3, legend=False, ax=ax)
ax.plot(data["incidence"].values, "ro-", linewidth=2, markersize=4, label="Observed")
ax.set_xlabel("Day")
ax.set_ylabel("Incidence")
ax.set_title("Simulated trajectories vs observed")
ax.legend()
ax.grid(linestyle=":")

plt.tight_layout()
plt.show()

print(f"Pending samples for iteration 2: {len(pending_2)}")
print(f"  beta  range: [{pending_2['beta'].min():.3f}, {pending_2['beta'].max():.3f}]")
print(f"  gamma range: [{pending_2['gamma'].min():.3f}, {pending_2['gamma'].max():.3f}]")

In [ ]:
# Accept the first iteration
engine.commit_step()
print(f"Iteration 1 committed. Current iteration: {engine.current_iteration}")

## Iteration 2

Switch to a different feature (an earlier time point) and run the second iteration.

In [ ]:
engine.update_feature_selection(["incidence_5"])
print("Feature selection updated to ['incidence_5']")

print("Running iteration 2...")
result_2 = engine.step()

print(f"  Samples generated:  {len(result_2.samples)}")
print(f"  Acceptance rate:    {engine.acceptance_rate:.3f}")
print(f"  Features emulated:  {result_2.selected_features}")

### Emulator Diagnostics for Iteration 2

In [ ]:
emulator_2 = result_2.get_emulator_for_feature("incidence_5")
emulator_2.test()
emulator_2.plot_diagnostics()

### Parameter Space After Iteration 2

In [ ]:
samples_2 = result_2.samples

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.scatter(samples_1["beta"], samples_1["gamma"], alpha=0.5, s=15, color="lightgray", label="After iter 1")
ax.scatter(samples_2["beta"], samples_2["gamma"], alpha=0.7, s=20, color="steelblue", label="After iter 2")
ax.axvline(beta_true,  color="red", linestyle="--", linewidth=2, label=f"True beta={beta_true}")
ax.axhline(gamma_true, color="red", linestyle="--", linewidth=2, label=f"True gamma={gamma_true}")
ax.set_xlabel("beta")
ax.set_ylabel("gamma")
ax.set_title("Parameter space evolution")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
sims_2 = sir_model_wrapper(samples_2)
sims_2.T.plot(color="steelblue", alpha=0.3, legend=False, ax=ax)
ax.plot(data["incidence"].values, "ro-", linewidth=2, markersize=4, label="Observed")
ax.set_xlabel("Day")
ax.set_ylabel("Incidence")
ax.set_title("Trajectories after 2 iterations")
ax.legend()
ax.grid(linestyle=":")

plt.tight_layout()
plt.show()

beta_ok  = samples_2["beta"].min()  <= beta_true  <= samples_2["beta"].max()
gamma_ok = samples_2["gamma"].min() <= gamma_true <= samples_2["gamma"].max()
print(f"True beta  in plausible region: {'Yes' if beta_ok  else 'No'}")
print(f"True gamma in plausible region: {'Yes' if gamma_ok else 'No'}")

In [ ]:
engine.commit_step()
print(f"Iteration 2 committed. Total iterations: {engine.current_iteration}")

## Final Summary

In [ ]:
print("FINAL HISTORY MATCHING RESULTS")
print("=" * 50)
print(f"True parameters:  beta={beta_true}, gamma={gamma_true}, R0={beta_true/gamma_true:.2f}")
print()
print(f"Plausible ranges after {engine.current_iteration} iterations:")
print(f"  beta:  [{samples_2['beta'].min():.3f}, {samples_2['beta'].max():.3f}]")
print(f"  gamma: [{samples_2['gamma'].min():.3f}, {samples_2['gamma'].max():.3f}]")
R0_final = samples_2["beta"] / samples_2["gamma"]
print(f"  R0:    [{R0_final.min():.2f}, {R0_final.max():.2f}]")
print()
print(f"Engine statistics:")
print(f"  Total samples generated: {engine.progress.total_samples_generated}")
print(f"  Total samples accepted:  {engine.progress.total_samples_accepted}")
print(f"  Final acceptance rate:   {engine.acceptance_rate:.3f}")
print(f"  Emulators trained:       {engine.progress.total_emulators_trained}")

beta_reduction  = (parameter_bounds["beta"][1]  - parameter_bounds["beta"][0])  / (samples_2["beta"].max()  - samples_2["beta"].min())
gamma_reduction = (parameter_bounds["gamma"][1] - parameter_bounds["gamma"][0]) / (samples_2["gamma"].max() - samples_2["gamma"].min())
print()
print(f"Parameter space reduced:")
print(f"  beta  space reduced by factor of {beta_reduction:.1f}x")
print(f"  gamma space reduced by factor of {gamma_reduction:.1f}x")

## Summary

This notebook demonstrated the **manual** history matching workflow:

1. **Setup**: imported `SIR` and `generate_observed_data` from `model.py`
2. **Iteration 1**: configured with a specific feature (`incidence_11`), ran `engine.step()`,
   inspected emulator diagnostics and parameter space, then committed with `engine.commit_step()`
3. **Iteration 2**: updated feature selection to `incidence_5`, repeated the step-inspect-commit
   cycle

Key API points:
- `engine.step()` runs one iteration and returns an `IterationResult`
- `engine.commit_step()` accepts the result and advances the iteration counter
- `engine.update_feature_selection(...)` changes features between iterations
- `result.get_emulator_for_feature(name)` gives access to emulator diagnostics

For the one-call automated workflow, see `01_basic_workflow.ipynb`.
For advanced configuration options, see `03_advanced_configuration.ipynb`.